In [8]:
import pandas as pd 

In [3]:
df = pd.read_csv('Train2.csv')

In [7]:
df.sample(5) 

,url,Phish?
70463,http://onebone.square.site,1
642,http://apsolutnadestrukcija.com/jaoo-pazi-ogro...,0
57304,http://docs.google.com/presentation/d/e/2PACX-...,1
82664,http://askgriff.com,1
110989,http://www.senshi-akademie.com/t876-die-prinze...,0


In [9]:
pip install tldextract


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tldextract]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import re
import tldextract
import math
from urllib.parse import urlparse
from collections import Counter

def shannon_entropy(string):
    if not string:
        return 0
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    entropy = -sum([p * math.log2(p) for p in prob])
    return entropy

def has_ip(hostname):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, hostname) else 0

def count_digits(s):
    return sum(c.isdigit() for c in s)

def count_letters(s):
    return sum(c.isalpha() for c in s)

def count_special_chars(s):
    # Counts characters that are not alphanumeric
    return len(re.findall(r'[^a-zA-Z0-9]', s))

def count_words(s):
    words = re.split(r'[\W_]+', s)
    words = [w for w in words if w]
    return len(words)

def is_shortened(url):
    pattern = r'bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|is\.gd|cli\.gs|tr\.im'
    return 1 if re.search(pattern, url.lower()) else 0

def keyword_count(url):
    suspicious_keywords = [
        'login', 'secure', 'update', 'bank', 'account',
        'verify', 'paypal', 'signin', 'confirm', 'free',
        'webscr', 'ebay', 'amazon', 'wallet', 'bonus'
    ]
    return sum(word in url.lower() for word in suspicious_keywords)


def extract_features(url):
    if not isinstance(url, str) or len(url) == 0:
        return None

    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    hostname = parsed.netloc
    domain = ext.domain
    subdomain = ext.subdomain
    path = parsed.path
    url_len = len(url)
    
    features = {}
    
    features['url_length'] = url_len
    features['domain_length'] = len(domain)
    features['subdomain_length'] = len(subdomain)
    features['path_length'] = len(path)
    
    features['dot_count'] = url.count('.')
    features['dash_count'] = url.count('-')
    features['underscore_count'] = url.count('_')
    features['slash_count'] = url.count('/')
    features['question_count'] = url.count('?')
    features['equal_count'] = url.count('=')
    features['at_count'] = url.count('@')
    features['hostname_dot_count'] = hostname.count('.')
    
    features['digit_count'] = count_digits(url)
    features['letter_count'] = count_letters(url)
    features['special_char_count'] = count_special_chars(url)
    
    features['digit_ratio'] = features['digit_count'] / url_len if url_len > 0 else 0
    features['letter_ratio'] = features['letter_count'] / url_len if url_len > 0 else 0
    
    features['entropy'] = shannon_entropy(url)
    features['has_ip'] = has_ip(hostname) # Checked against hostname specifically
    features['is_shortened'] = is_shortened(url)
    
    # Check if 'www' appears anywhere except the very start
    features['bad_www'] = 1 if 'www' in url.lower() and not url.lower().startswith('http://www') else 0
    
    # Check if a common TLD is hidden in the path (e.g., example.com/paypal.com/)
    tld_in_path_pattern = r'\.com/|\.net/|\.org/|\.edu/|\.gov/'
    features['tld_in_path'] = 1 if re.search(tld_in_path_pattern, path.lower()) else 0
    
    features['word_count'] = count_words(url)
    features['keyword_count'] = keyword_count(url)
    
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'pw']
    features['suspicious_tld'] = 1 if ext.suffix in suspicious_tlds else 0
    
    return features


def build_feature_dataframe(df, url_column):
    
    feature_list = df[url_column].apply(lambda x: extract_features(x))
    
    feature_df = pd.DataFrame(feature_list.tolist())
    
    feature_df = feature_df.fillna(0)
    
    return feature_df

In [12]:
feature_matrix = build_feature_dataframe(df, 'url')


Starting feature extraction for 111401 URLs...
Feature extraction complete.


In [13]:
feature_matrix

,url_length,domain_length,subdomain_length,path_length,dot_count,dash_count,underscore_count,slash_count,question_count,equal_count,...,digit_ratio,letter_ratio,entropy,has_ip,is_shortened,bad_www,tld_in_path,word_count,keyword_count,suspicious_tld
0,36,15,0,10,1,0,0,4,0,0,...,0.000000,0.833333,3.993133,0,0,0,0,5,0,0
1,45,4,8,18,4,0,1,3,0,0,...,0.022222,0.777778,4.402530,0,0,0,0,8,0,0
2,20,3,0,7,1,1,0,3,0,0,...,0.050000,0.650000,3.746439,0,0,0,0,5,0,0
3,60,16,3,20,3,1,0,4,1,1,...,0.033333,0.766667,4.460324,0,0,0,0,11,0,0
4,174,6,4,108,2,1,0,7,1,3,...,0.109195,0.781609,5.574198,0,0,0,0,18,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111396,147,11,0,125,1,16,0,6,0,0,...,0.054422,0.782313,4.445113,0,0,0,0,23,0,0
111397,32,7,3,8,3,0,0,3,0,0,...,0.000000,0.781250,3.905639,0,0,0,0,6,0,0
111398,50,6,3,29,2,1,0,5,0,0,...,0.040000,0.780000,4.351272,0,0,0,0,8,0,0
111399,20,6,0,3,1,0,0,3,0,0,...,0.000000,0.750000,3.608695,0,0,0,0,4,0,1


In [18]:
final_data = pd.concat([feature_matrix, df['Phish?']], axis=1)

In [19]:
final_data 

,url_length,domain_length,subdomain_length,path_length,dot_count,dash_count,underscore_count,slash_count,question_count,equal_count,...,letter_ratio,entropy,has_ip,is_shortened,bad_www,tld_in_path,word_count,keyword_count,suspicious_tld,Phish?
0,36,15,0,10,1,0,0,4,0,0,...,0.833333,3.993133,0,0,0,0,5,0,0,0
1,45,4,8,18,4,0,1,3,0,0,...,0.777778,4.402530,0,0,0,0,8,0,0,0
2,20,3,0,7,1,1,0,3,0,0,...,0.650000,3.746439,0,0,0,0,5,0,0,1
3,60,16,3,20,3,1,0,4,1,1,...,0.766667,4.460324,0,0,0,0,11,0,0,0
4,174,6,4,108,2,1,0,7,1,3,...,0.781609,5.574198,0,0,0,0,18,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111396,147,11,0,125,1,16,0,6,0,0,...,0.782313,4.445113,0,0,0,0,23,0,0,0
111397,32,7,3,8,3,0,0,3,0,0,...,0.781250,3.905639,0,0,0,0,6,0,0,0
111398,50,6,3,29,2,1,0,5,0,0,...,0.780000,4.351272,0,0,0,0,8,0,0,0
111399,20,6,0,3,1,0,0,3,0,0,...,0.750000,3.608695,0,0,0,0,4,0,1,1


In [28]:
final_data[(final_data['is_shortened'] == 1) & (final_data['Phish?']== 1 )].size

111670

In [29]:
final_data[(final_data['is_shortened'] == 1) & (final_data['Phish?']== 0 )].size

199732

In [31]:
final_data.to_csv('Final.csv' , index = False )

In [10]:
from sklearn.linear_model import LogisticRegression

In [6]:
from sklearn.model_selection import train_test_split

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = df.iloc[:, :-1]
Y = df.iloc[:, -1]
X = scaler.fit_transform(X)

In [13]:
X_train, X_test , y_train , y_test = train_test_split(X ,Y, test_size = 0.2 , random_state = 42 ) 

In [25]:
X_train.shape

(89120, 25)

In [26]:
X_test.shape

(22281, 25)

In [27]:
y_train.shape

(89120,)

In [28]:
y_test.shape

(22281,)

In [29]:
LR = LogisticRegression()

In [30]:
LR.fit(X_train, y_train ) 

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [31]:
y_pred = LR.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score

In [33]:
accuracy_score(y_test  , y_pred ) 

0.8535972353125981

In [34]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [35]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.9556572864772677

In [14]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

In [15]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.951124276289215